In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#data classes
import xarray as xr

#arrays
import numpy as np

#loading bar
from tqdm import tqdm

#plotting 
import matplotlib.pyplot as plt

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
#SETUP SIMULATION REGION

# region = "TRACER"
# spinup_hours = "0"
# RunType = (region,"WET","NSSL",spinup_hours)

# spinup_hours = "-5"
# RunType = (region,"DIURNAL","NSSL",spinup_hours)

region = "Hawaii"
spinup_hours = "0"
RunType = (region,"TRADES","NSSL",spinup_hours)

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data", region)
dataType = "RadarData"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
########################
#CODE INFORMATION

In [ ]:
#Getting Data
# https://projectpythia.org/mrms-cookbook/notebooks/ch4-realtimedata/
# Authors:
# Ty Janoski
# City College of New York and NOAA/OAR National Severe Storms Laboratory
# Mya Sears
# NSF National Center for Atmospheric Research
# Bella Condo
# University at Albany (State University of New York)
# JD Heaton
# Metropolitan State University of Denver
# MaKenna Collins
# Jackson State University
# Maxwell Grover
# Argonne National Laboratory

In [ ]:
########################
#LIBRARIES

In [ ]:
#system packages
import re

#data loading packages
import pandas as pd

# Packages required to request and open data from AWS S3
import s3fs
import urllib
import tempfile
import gzip

In [ ]:
########################
#DATA RETRIEVAL FUNCTIONS

In [ ]:
# ============================================================
# Helper: extract time from filename
# ============================================================
def extract_time_from_filename(fname):
    """Parse MRMS filename and return a pandas.Timestamp (UTC)."""
    match = re.search(r"_(\d{8}-\d{6})\.grib2\.gz", fname)
    return pd.to_datetime(match.group(1), format="%Y%m%d-%H%M%S", utc=True)

# ============================================================
# Select target times at 6-hour intervals
# ============================================================
def select_nearest_to_targets(files, interval_hours="6h"):
    """Return subset of file paths nearest to each N-hour UTC time."""
    # Extract actual timestamps from filenames
    file_times = pd.Series({f: extract_time_from_filename(f) for f in files}).sort_values()

    # Build list of desired UTC times covering file range
    start = file_times.min().floor(interval_hours)   # lowercase h
    end   = file_times.max().ceil(interval_hours)
    targets = pd.date_range(start, end, freq=interval_hours, tz="UTC")
    

    selected = []
    for t in targets:
        nearest_idx = (abs(file_times - t)).argmin()
        selected.append(file_times.index[nearest_idx])

    return selected

# def RetrieveMRMSRadarData_V1(ModelData, region, product, datestrings,
#                              interval_hours="6h",
#                              outputPath=""):
#     """
#     Retrieve MRMS reflectivity data and save each selected timestep
#     as an individual NetCDF file.
#     """
#     aws = s3fs.S3FileSystem(anon=True)

#     for datestring in datestrings:
#         print(f"\n=== Processing {datestring} ===")

#         # List all files for this date
#         try:
#             data_files = aws.ls(f'noaa-mrms-pds/{region}/{product}/{datestring}/')
#         except Exception as e:
#             print(f"Could not access {datestring}: {e}")
#             continue

#         if not data_files:
#             print(f"No files found for {datestring}")
#             continue

#         data_files = sorted(data_files)
#         selected_files = select_nearest_to_targets(data_files, interval_hours=interval_hours)

#         for f in tqdm(selected_files, desc=f"{datestring} files", leave=False, unit="file"):

#             try:
#                 response = urllib.request.urlopen(f"https://noaa-mrms-pds.s3.amazonaws.com/{f[14:]}")
#                 compressed_file = response.read()

#                 with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
#                     tmp.write(gzip.decompress(compressed_file))
#                     tmp.flush()
#                     ds = xr.load_dataarray(tmp.name, engine="cfgrib", decode_timedelta=True)
#                     ds.name = product

#                     # Drop differing coords
#                     for coord in ["valid_time", "step"]:
#                         if coord in ds.coords:
#                             ds = ds.drop_vars(coord)

#                     # Attach true timestamp from filename
#                     file_time = extract_time_from_filename(f)
#                     ds = ds.expand_dims(time=[file_time])

#                     # Subset
#                     subset = DataSubsetting_Class.SubsetDataRegion(ds, ModelData)

#                     # Convert to timezone-naive time
#                     subset = subset.assign_coords(
#                         time=pd.to_datetime(subset.time.to_pandas()).tz_localize(None)
#                     )

#                     # --- Save each timestep as its own NetCDF ---
#                     time_str = file_time.strftime("%Y%m%d-%H%M%S")
#                     outputFile = f"MRMSReflectivity_{region}_{ModelData.region}_{time_str}.nc"
#                     os.makedirs(outputPath, exist_ok=True)
#                     outputFilePath = os.path.join(outputPath, outputFile)

#                     subset.to_netcdf(outputFilePath)
#                     print(f"Saved: {outputFilePath}")
 
#             except Exception as e:
#                 print(f"Failed to load {f}: {e}")
#                 continue

def RetrieveMRMSRadarData_V2(ModelData, region, product, datestrings,
                             interval_hours="6h",
                             outputPath=""):
    """
    Retrieve MRMS reflectivity data and save each selected timestep
    as an individual NetCDF file.
    """
    aws = s3fs.S3FileSystem(anon=True)

    for datestring in datestrings:
        print(f"\n=== Processing {datestring} ===")

        # List all files for this date
        try:
            data_files = aws.ls(f'noaa-mrms-pds/{region}/{product}/{datestring}/')
        except Exception as e:
            print(f"Could not access {datestring}: {e}")
            continue

        if not data_files:
            print(f"No files found for {datestring}")
            continue

        data_files = sorted(data_files)
        selected_files = select_nearest_to_targets(data_files, interval_hours=interval_hours)

        if datestring == datestrings[-1]:
            print("Final date detected — using only the first MRMS file.")
            selected_files = selected_files[:1]

        for f in tqdm(selected_files, desc=f"{datestring} files", leave=False, unit="file"):

            # ------------------------------------------------------------------
            # Build output filename BEFORE downloading anything
            # ------------------------------------------------------------------
            file_time = extract_time_from_filename(f)
            time_str = file_time.strftime("%Y%m%d-%H%M%S")

            dataType = "Reflectivity" if "MergedReflectivityQC" in product else "QPE"
            outputFile = f"MRMS{dataType}_{region}_{ModelData.region}_{time_str}.nc"
            os.makedirs(outputPath, exist_ok=True)
            outputFilePath = os.path.join(outputPath, outputFile)

            # ------------------------------------------------------------------
            # Skip if this file already exists
            # ------------------------------------------------------------------
            if os.path.exists(outputFilePath):
                print(f"Already exists, skipping: {outputFilePath}")
                continue

            # ------------------------------------------------------------------
            # Otherwise download + process
            # ------------------------------------------------------------------
            try:
                response = urllib.request.urlopen(
                    f"https://noaa-mrms-pds.s3.amazonaws.com/{f[14:]}"
                )
                compressed_file = response.read()

                with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
                    tmp.write(gzip.decompress(compressed_file))
                    tmp.flush()
                    ds = xr.load_dataarray(tmp.name, engine="cfgrib", decode_timedelta=True)
                    ds.name = product

                # Drop differing coords
                for coord in ["valid_time", "step"]:
                    if coord in ds.coords:
                        ds = ds.drop_vars(coord)

                # Attach true timestamp
                ds = ds.expand_dims(time=[file_time])

                # Subset to model domain
                subset = DataSubsetting_Class.SubsetDataRegion(ds, ModelData)

                # Convert to timezone-naive
                subset = subset.assign_coords(
                    time=pd.to_datetime(subset.time.to_pandas()).tz_localize(None)
                )

                # Save
                subset.to_netcdf(outputFilePath)
                print(f"Saved: {outputFilePath}")

            except Exception as e:
                print(f"Failed to load {f}: {e}")
                continue


In [ ]:
#########################
#LIST AVAILABLE DATA
def ListAllMRMSProducts(region="CONUS"):
    """
    Returns a list of all MRMS products available for a given region.
    """
    aws = s3fs.S3FileSystem(anon=True)

    # List directory
    products = aws.ls(f"noaa-mrms-pds/{region}/")
    products = [p.split("/")[-1] for p in products]
    return products

all_products = ListAllMRMSProducts("CONUS")
# print(all_products)

In [ ]:
def SearchMRMSProducts(key, region="CONUS"):
    """
    Returns all MRMS products whose names contain the given key.
    """
    all_products = ListAllMRMSProducts(region)
    matches = [p for p in all_products if key in p]
    return matches

#Getting List of Product Options
# https://data.ucar.edu/en/dataset/mrms-merged-base-reflectivity-quality-controlled-data
# https://data.eol.ucar.edu/dataset/577.002

matches = SearchMRMSProducts("MergedReflectivityQC_", "CONUS")
matches += ["MultiSensor_QPE_01H_Pass2_00.00"]
print(matches)

In [ ]:
levels = [match[21:] for match in matches[:-1]]
levels = [float(level) for level in levels]
print(levels)
plt.plot(levels,color='blue',label="MRMS"); plt.ylabel("Altitude (km)"); plt.ylabel("Index")

z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
zlevels = np.loadtxt(z_levels_filePath)/1e3
zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
a = np.where(zlevels_center>np.max(levels))[0][0]
plt.plot(zlevels_center[:a],color='black',label="MPAS")
plt.legend()

In [ ]:
########################
#DATA RETRIEVAL

In [ ]:
def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates

In [ ]:
# ============================================================
# Main Loop: Load multiple dates
# ============================================================
region_options = [
    "CONUS",
    "ALASKA",
    "CARIB",
    "GUAM",
    "HAWAII"
]

product_options = matches

# Retrieve the user selection from 'Region' 
region = region_options[0] if ModelData.region=="TRACER" else region_options[-1]

# Retrieve the user selection from 'MRMS product'

simulationDates = ModelData.simulationDates
simulationDates = CorrectSimulationDates(ModelData, simulationDates)

datestrings = [datestring.replace('-','') for datestring in simulationDates]


for product in product_options:
    outputPath = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates[0]}_{simulationDates[-1]}",product)
    
    MRMSData = RetrieveMRMSRadarData_V2(ModelData,region,product,datestrings,
                                     interval_hours="15min",
                                     outputPath = outputPath)